## Ingest circuits.csv


### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"

In [0]:
dbutils.widgets.text("p_source_value", "")
v_source_value = dbutils.widgets.get("p_source_value")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp


### Step 1 - Read the csv file


In [ ]:
circuits_path = f"{raw_folder_path}/circuits.csv"
circuits_path


In [0]:
circuits_df = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv(circuits_path)


#### 1.1 Define the schema


In [0]:
circuits_schema = StructType([
    StructField("circuitId", IntegerType(), False),
    StructField("circuitRef", StringType(), True),
    StructField("name", StringType(), True),
    StructField("location", StringType(), True),
    StructField("country", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lng", DoubleType(), True),
    StructField("alt", IntegerType(), True),
    StructField("url", StringType(), True)
])


#### 1.2 Read the csv file


In [0]:
circuits_df = spark.read \
    .option('header', True) \
    .schema(circuits_schema) \
    .csv(circuits_path)


In [0]:
circuits_df.printSchema()


### Step 2 - Transform the data


1. Only allow you to select the columns
    


In [0]:
circuits_selected_df = circuits_df.select(
    "circuitId",
    "circuitRef",
    "name",
    "location",
    "country",
    "lat",
    "lng",
    "alt"
)


2. The following 3 allow you to use column methods


In [0]:
circuits_selected_df = circuits_df.select(
    circuits_df.circuitId,
    circuits_df.circuitRef,
    circuits_df.name,
    circuits_df.location,
    circuits_df.country,
    circuits_df.lat,
    circuits_df.lng,
    circuits_df.alt
)


In [0]:
circuits_selected_df = circuits_df.select(
    circuits_df["circuitId"],
    circuits_df["circuitRef"],
    circuits_df["name"],
    circuits_df["location"],
    circuits_df["country"],
    circuits_df["lat"],
    circuits_df["lng"],
    circuits_df["alt"]
)


In [0]:
circuits_selected_df = circuits_df.select(
    col("circuitId"),
    col("circuitRef"),
    col("name"),
    col("location"),
    col("country"),
    col("lat"),
    col("lng"),
    col("alt")
)


In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("circuitRef", "circuit_ref")
        .withColumnRenamed("lat", "latitude")
        .withColumnRenamed("lng", "longitud")
        .withColumnRenamed("alt", "altitude")
        .withColumn("source", lit(v_source_value))
)


In [0]:
circuits_final_df = add_ingestion_date(circuits_renamed_df)


### Step 3 - Write the output to parquet


In [0]:
circuits_final_df.write.mode("overwrite").format("delta").saveAsTable(f"f1_processed.circuits")


In [0]:
circuits_df_proccesed = spark.read.table("f1_processed.circuits")
circuits_df_proccesed.display()


In [0]:
dbutils.notebook.exit("success")